# PREDICT-Play | 02. Feature engineering

Run this notebook after `01_preprocessing.ipynb`.


The 40% platform pool is divided into four disjoint roles:
- 25% reference titles: used only to construct historical genre summaries.
- 50% supervised training titles.
- 12.5% tuning titles.
- 12.5% probability-calibration titles.

The 60% acquisition catalogue has no target available to model development and is scored only after the models are fitted.

## 1. Imports and data locations

In [2]:
# Import the libraries used for feature engineering.
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Load the preprocessing outputs

In [33]:
# Load only the tables needed for model feature construction.
platform_catalogue = pd.read_csv("https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/preprocessed_data/platform_catalogue.csv")
acquisition_catalogue = pd.read_csv("https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/preprocessed_data/acquisition_cat.csv")
outcomes = pd.read_csv("https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/preprocessed_data/platform_title_outcomes.csv")

In [4]:
# Keep only the target from the outcome table.

platform = platform_catalogue.merge(outcomes[["movieId", "target"]], on="movieId", how="inner",validate="one_to_one")
platform["target"] = platform["target"].astype(int)


,titles
target,
0,8345
1,2566


## 3. Create the development roles

The split is performed at title level and is stratified on the platform target. Reference titles are not used in either model's supervised loss; their outcomes are used only to construct historical genre-level analogue features.

In [5]:
# Reserve 25% of platform titles as the fixed historical reference library.
reference_titles, remaining_titles = train_test_split(platform,train_size=0.25,random_state=42,stratify=platform["target"])


In [6]:
# Use two thirds of the remaining titles for supervised model training.
# This is 50% of the original platform pool.
train_titles, remaining_holdout = train_test_split(remaining_titles,train_size=2 / 3,random_state=42,stratify=remaining_titles["target"])


In [7]:
# Split the remaining 25% of platform titles equally between tuning and calibration.
tune_titles, calibration_titles = train_test_split(remaining_holdout,train_size=0.50,random_state=42,stratify=remaining_holdout["target"])


In [8]:
# Assign one role to every platform title.
platform["role"] = ""

platform.loc[platform["movieId"].isin(reference_titles["movieId"]),"role"] = "reference"
platform.loc[platform["movieId"].isin(train_titles["movieId"]),"role"] = "train"
platform.loc[platform["movieId"].isin(tune_titles["movieId"]),"role"] = "tune"
platform.loc[platform["movieId"].isin(calibration_titles["movieId"]),"role"] = "calibration"

# Acquisition titles have no development target.
acquisition = acquisition_catalogue.copy()
acquisition["target"] = -1
acquisition["role"] = "catalogue"

all_titles = pd.concat([platform, acquisition],ignore_index=True,sort=False)

# Keep a stable movie-node ordering for the graph notebook.
all_titles = all_titles.sort_values("movieId").reset_index(drop=True)
all_titles["node_id"] = np.arange(len(all_titles))


,titles
role,
calibration,1364
catalogue,16367
reference,2727
train,5456
tune,1364


## 4. Build reference-only historical genre features

Only outcomes from the fixed reference library are used here. For each genre we calculate a smoothed historical success rate. Titles with genres that have no reference support fall back to the overall reference success rate.

This gives both models a historical analogue signal without using the candidate title's own viewing outcome.

In [9]:
# Clean the MovieLens genre string before exploding it into title-genre rows.
all_titles["genres"] = all_titles["genres"].fillna("(no genres listed)")
platform["genres"] = platform["genres"].fillna("(no genres listed)")

reference_for_genres = platform[platform["role"] == "reference"][["movieId", "genres", "target"]].copy()
reference_genres = reference_for_genres.assign(genre=reference_for_genres["genres"].str.split("|")).explode("genre")
reference_genres["genre"] = reference_genres["genre"].str.strip()


In [10]:
# Calculate the number of reference titles and successes associated with each genre.
genre_reference_stats = (
    reference_genres
    .groupby("genre", as_index=False)
    .agg(
        reference_titles=("movieId", "nunique"),
        reference_successes=("target", "sum")
    )
)

reference_success_rate = reference_for_genres["target"].mean()
smoothing_titles = 5.0

genre_reference_stats["reference_success_rate"] = (genre_reference_stats["reference_successes"]+ smoothing_titles * reference_success_rate) / ( genre_reference_stats["reference_titles"]+ smoothing_titles)


,genre,reference_titles,reference_successes,reference_success_rate
8,Drama,1284,306,0.238305
5,Comedy,843,235,0.278509
15,Romance,400,112,0.279445
17,Thriller,399,151,0.376671
1,Action,371,148,0.396743
6,Crime,275,91,0.329197
7,Documentary,270,8,0.033365
11,Horror,269,70,0.259764
2,Adventure,223,83,0.369190
16,Sci-Fi,171,67,0.387360


In [11]:
# Attach the reference genre statistics to every title, including acquisition candidates.
title_genres = all_titles[["movieId", "genres"]].copy()

title_genres["genre"] = title_genres["genres"].str.split("|")
title_genres = title_genres.explode("genre")
title_genres["genre"] = title_genres["genre"].str.strip()

title_genres = title_genres.merge(genre_reference_stats[["genre","reference_titles","reference_success_rate"]],on="genre",how="left")

title_genres["genre_supported_by_reference"] = (title_genres["reference_titles"].notna().astype(int))
title_genres["reference_titles"] = (title_genres["reference_titles"].fillna(0))
title_genres["reference_success_rate"] = (title_genres["reference_success_rate"].fillna(reference_success_rate))


In [12]:
# Aggregate the reference evidence across each title's genres.
title_reference_features = (
title_genres
    .groupby("movieId", as_index=False)
    .agg(
        reference_genre_success_mean=("reference_success_rate", "mean"),
        reference_genre_success_max=("reference_success_rate", "max"),
        reference_genre_support=("reference_titles", "sum"),
        reference_genre_coverage=("genre_supported_by_reference", "mean")
    )
)

all_titles = all_titles.merge(title_reference_features,on="movieId",how="left",validate="one_to_one")


## 5. Create shared tabular features

The feature set uses MovieLens genre and release-year information plus reference-only historical genre summaries. `tmdb_available` is retained as a metadata-coverage indicator.

The following fields are intentionally **not** model inputs:
- `tmdb_popularity`, because it was used to generate the synthetic target.
- `observed_households`, `qualifying_households`, watch duration or viewing counts, because they directly define or reveal the outcome.
- synthetic search counts, because the search generator was independent of observed demand.
- licence dates and synthetic licence costs.

In [34]:
# Prepare simple numeric title features.
all_titles["release_year"] = pd.to_numeric(all_titles["release_year"],errors="coerce")
all_titles["release_year_missing"] = (all_titles["release_year"].isna().astype(int))
train_year_median = all_titles.loc[all_titles["role"] == "train","release_year"].median()
all_titles["release_year_filled"] = (all_titles["release_year"].fillna(train_year_median))
all_titles["title_age_2026"] = (2026 - all_titles["release_year_filled"]).clip(lower=0)
all_titles["genre_count"] = (all_titles["genres"].str.split("|").str.len())

# convert tmbd avilable to a int flag
all_titles["tmdb_available_flag"] = (all_titles["tmdb_available"].astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0}).fillna(0).astype(int))


In [14]:
# One-hot encode the MovieLens genres for both models.
genre_dummies = all_titles["genres"].str.get_dummies(sep="|")

genre_dummies.columns = ["genre_" + re.sub(r"[^A-Za-z0-9]+", "_", column).strip("_").lower() for column in genre_dummies.columns]
genre_dummies = genre_dummies.astype(float)



Genre features: 20


In [16]:
# Standardise numeric features using training titles only.
numeric_columns = [
    "release_year_filled",
    "title_age_2026",
    "release_year_missing",
    "genre_count",
    "reference_genre_success_mean",
    "reference_genre_success_max",
    "reference_genre_support",
    "reference_genre_coverage",
    "tmdb_available_flag"
]

scaler = StandardScaler()

scaler.fit(
    all_titles.loc[
        all_titles["role"] == "train",
        numeric_columns
    ]
)

scaled_numeric = pd.DataFrame(
    scaler.transform(all_titles[numeric_columns]),
    columns=[f"scaled_{column}" for column in numeric_columns],
    index=all_titles.index
)

In [18]:
# Build one model-ready title table shared by XGBoost and the R-GCN movie nodes.
identifier_columns = [
"node_id",
    "movieId",
    "title_id",
    "canonical_title",
    "role",
    "target"
]

model_features = pd.concat([all_titles[identifier_columns].reset_index(drop=True),scaled_numeric.reset_index(drop=True),genre_dummies.reset_index(drop=True)],axis=1)


Model feature table: (27278, 35)


## 6. Build the visible R-GCN relation graph

The graph contains three node types: movies, genres and release decades. Four directed relation types are used:

- `movie_has_genre`
- `genre_has_movie`
- `movie_released_in_decade`
- `decade_has_movie`

The graph uses only metadata available for both platform and acquisition titles. It does not contain TMDB popularity or catalogue outcomes.

In [19]:
# Create release-decade values used as a second graph relationship.
all_titles["release_decade"] = ((all_titles["release_year_filled"] // 10) * 10).astype(int)

movie_nodes = all_titles[["node_id", "movieId"]].copy()

genre_names = sorted(title_genres["genre"].dropna().unique().tolist())

decade_names = sorted(all_titles["release_decade"].unique().tolist())

first_genre_node = len(movie_nodes)
first_decade_node = first_genre_node + len(genre_names)

genre_nodes = pd.DataFrame({
    "genre": genre_names,
    "node_id": np.arange(
        first_genre_node,
        first_genre_node + len(genre_names)
    )
})

decade_nodes = pd.DataFrame({
    "release_decade": decade_names,
    "node_id": np.arange(
        first_decade_node,
        first_decade_node + len(decade_names)
    )
})


Movie nodes: 27278
Genre nodes: 20
Decade nodes: 13


In [20]:
# Build movie-to-genre edges.
movie_genre_edges = title_genres[["movieId", "genre"]].merge(
    movie_nodes,
    on="movieId",
    how="left"
).merge(
    genre_nodes,
    on="genre",
    how="left",
    suffixes=("_movie", "_genre")
)

movie_to_genre = pd.DataFrame({
    "source": movie_genre_edges["node_id_movie"],
    "target": movie_genre_edges["node_id_genre"],
    "relation": "movie_has_genre"
})

genre_to_movie = pd.DataFrame({
    "source": movie_genre_edges["node_id_genre"],
    "target": movie_genre_edges["node_id_movie"],
    "relation": "genre_has_movie"
})


In [21]:
# Build movie-to-release-decade edges.
movie_decade_edges = all_titles[[
    "node_id",
    "release_decade"
]].merge(
    decade_nodes,
    on="release_decade",
    how="left",
    suffixes=("_movie", "_decade")
)

movie_to_decade = pd.DataFrame({
    "source": movie_decade_edges["node_id_movie"],
    "target": movie_decade_edges["node_id_decade"],
    "relation": "movie_released_in_decade"
})

decade_to_movie = pd.DataFrame({
    "source": movie_decade_edges["node_id_decade"],
    "target": movie_decade_edges["node_id_movie"],
    "relation": "decade_has_movie"
})


In [23]:
# Combine and save all graph edges.
rgcn_edges = pd.concat([movie_to_genre,genre_to_movie,movie_to_decade,decade_to_movie],ignore_index=True)

rgcn_edges[["source", "target"]] = rgcn_edges[["source","target"]].astype(int)



,edges
relation,
movie_has_genre,54406
genre_has_movie,54406
movie_released_in_decade,27278
decade_has_movie,27278


In [25]:
# Create graph-node features.

shared_feature_columns = [ column for column in model_features.columns if column not in identifier_columns]

movie_node_features = model_features[["node_id"] + shared_feature_columns].copy()
movie_node_features["node_type_movie"] = 1.0
movie_node_features["node_type_genre"] = 0.0
movie_node_features["node_type_decade"] = 0.0
movie_node_features["node_type"] = "movie"
movie_node_features["entity_name"] = all_titles["canonical_title"].astype(str)

metadata_feature_columns = shared_feature_columns + [
    "node_type_movie",
    "node_type_genre",
    "node_type_decade"
]

genre_node_features = pd.DataFrame(
    0.0,
    index=np.arange(len(genre_nodes)),
    columns=metadata_feature_columns
)
genre_node_features["node_id"] = genre_nodes["node_id"].to_numpy()
genre_node_features["node_type_genre"] = 1.0
genre_node_features["node_type"] = "genre"
genre_node_features["entity_name"] = genre_nodes["genre"].astype(str).to_numpy()

decade_node_features = pd.DataFrame(
    0.0,
    index=np.arange(len(decade_nodes)),
    columns=metadata_feature_columns
)
decade_node_features["node_id"] = decade_nodes["node_id"].to_numpy()
decade_node_features["node_type_decade"] = 1.0
decade_node_features["node_type"] = "decade"
decade_node_features["entity_name"] = decade_nodes["release_decade"].astype(str).to_numpy()

rgcn_node_features = pd.concat(
    [
        movie_node_features,
        genre_node_features,
        decade_node_features
    ],
    ignore_index=True,
    sort=False
)

rgcn_node_features = rgcn_node_features.sort_values("node_id").reset_index(drop=True)


R-GCN node feature table: (27311, 35)


## 7. Final checks and outputs

In [36]:
model_features.to_csv("model_features.csv", index=False)
genre_reference_stats.to_csv("genre_reference_stats.csv", index=False)
rgcn_edges.to_csv("rgcn_edges.csv", index=False)
rgcn_node_features.to_csv("rgcn_node_features.csv", index=False)